# Adım 2 — Kafka ile Streaming Veri Üretimi

Bu notebook, Kafka Producer'ın nasıl çalıştığını açıklar ve Kafka'dan gelen mesajları test amaçlı okur.

---

## 2.1 Producer Mimarisi

Kafka Producer ayrı bir Docker container olarak çalışır (`kafka-producer` servisi).

**İş Akışı:**
1. NYC Yellow Taxi parquet dosyasını okur
2. 22 Ocak 2024 ve sonrasına filtreler
3. Her satırı JSON formatına çevirir
4. Saniyede ~200 kayıt hızında `taxi_trips` topic'ine gönderir

## 2.2 Producer Özellikleri

| Özellik | Değer |
|---------|-------|
| Gönderim hızı | ~200 kayıt/saniye |
| Mesaj formatı | JSON (UTF-8) |
| Topic adı | `taxi_trips` |
| Veri kaynağı | `yellow_tripdata_2024-01.parquet` |
| Filtre | pickup >= 2024-01-22 |
| Hata yönetimi | 5 denemeye kadar yeniden bağlanma |
| Ack ayarı | `all` (tam güvenilirlik) |

## 2.3 Producer Loglarını İzleme

```bash
docker-compose logs -f kafka-producer
```

## 2.4 Kafka'dan Test Amaçlı Mesaj Okuma

In [6]:
import json
from kafka import KafkaConsumer

KAFKA_BOOTSTRAP_SERVERS = "kafka:29092"
KAFKA_TOPIC = "taxi_trips"

In [7]:
consumer = KafkaConsumer(
    KAFKA_TOPIC,
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    group_id="test-consumer-group",
    value_deserializer=lambda m: json.loads(m.decode("utf-8")),
    consumer_timeout_ms=10000
)

print(f"Consumer '{KAFKA_TOPIC}' topic'ine baglandi.")
print(f"Ilk 5 mesaji okuyoruz...\n")

Consumer 'taxi_trips' topic'ine baglandi.
Ilk 5 mesaji okuyoruz...



In [8]:
count = 0
for message in consumer:
    count += 1
    print(f"--- Mesaj #{count} (offset: {message.offset}) ---")
    print(json.dumps(message.value, indent=2, ensure_ascii=False))
    print()
    if count >= 5:
        break

consumer.close()
print(f"\nToplam {count} mesaj okundu.")

--- Mesaj #1 (offset: 5) ---
{
  "VendorID": 2,
  "tpep_pickup_datetime": "2024-01-22T00:55:07",
  "tpep_dropoff_datetime": "2024-01-22T01:00:33",
  "passenger_count": 1.0,
  "trip_distance": 1.04,
  "RatecodeID": 1.0,
  "store_and_fwd_flag": "N",
  "PULocationID": 186,
  "DOLocationID": 68,
  "payment_type": 2,
  "fare_amount": 7.2,
  "extra": 1.0,
  "mta_tax": 0.5,
  "tip_amount": 0.0,
  "tolls_amount": 0.0,
  "improvement_surcharge": 1.0,
  "total_amount": 12.2,
  "congestion_surcharge": 2.5,
  "Airport_fee": 0.0
}

--- Mesaj #2 (offset: 6) ---
{
  "VendorID": 2,
  "tpep_pickup_datetime": "2024-01-22T00:38:47",
  "tpep_dropoff_datetime": "2024-01-22T01:08:02",
  "passenger_count": 1.0,
  "trip_distance": 18.86,
  "RatecodeID": 2.0,
  "store_and_fwd_flag": "N",
  "PULocationID": 132,
  "DOLocationID": 79,
  "payment_type": 1,
  "fare_amount": 70.0,
  "extra": 0.0,
  "mta_tax": 0.5,
  "tip_amount": 10.0,
  "tolls_amount": 0.0,
  "improvement_surcharge": 1.0,
  "total_amount": 85.75,
 

## 2.5 Topic İstatistikleri

In [9]:
from kafka import KafkaConsumer, TopicPartition

consumer = KafkaConsumer(
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    group_id="stats-group"
)

partitions = consumer.partitions_for_topic(KAFKA_TOPIC)
if partitions:
    total_messages = 0
    for p in partitions:
        tp = TopicPartition(KAFKA_TOPIC, p)
        consumer.assign([tp])
        consumer.seek_to_end(tp)
        end_offset = consumer.position(tp)
        consumer.seek_to_beginning(tp)
        begin_offset = consumer.position(tp)
        count = end_offset - begin_offset
        total_messages += count
        print(f"  Partition {p}: {count:,} mesaj")
    print(f"\nToplam: {total_messages:,} mesaj")
else:
    print(f"'{KAFKA_TOPIC}' topic'i bulunamadi.")

consumer.close()

  Partition 0: 53,600 mesaj

Toplam: 53,600 mesaj


## 2.6 Kafka UI

[http://localhost:8080](http://localhost:8080) adresinden topic, mesaj ve consumer izlenebilir.